## Reading 38-1 - Characterizing Noise

By the end of this reading, you will learn:
<ol>
    <li>How to measure a sensor's noise before deciding how to filter it</li>
    <li>What the mean, variance, and standard deviation tell you about a sensor</li>
    <li>Two simple filters, the moving average and exponential smoothing</li>
    <li>The trade-off every filter makes between noise and lag</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading38
    cd reading38

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec38/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    calibration.csv  innovation.c  kalman.c  Makefile  mission.csv  noise_stats.c  smoothing.c

### Measure Before You Filter

In Lecture 37, a leaky bucket kept noise from triggering a fault response, and hysteresis kept noise from switching a heater. Both had settings, a trip level and a dead band, and both readings said the right settings depend on <i>how noisy the sensor is</i>. Ama's slide on noise puts it first: <b>characterize sensor noise before choosing a filter</b>.

The way to do that is a <b>calibration</b> run: hold the sensor at a value you already know, record it many times, and study the differences. Every difference from the known value is noise.

### Mean, Variance, and Standard Deviation - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec38/noise_stats.c">noise_stats.c</a>

<code>calibration.csv</code> contains 200 readings from a temperature sensor held at exactly 20.00 C. The program computes three numbers:
<ul>
    <li>The <b>mean</b> is the average reading. If it differs from the known value, the sensor has a <b>bias</b>: it is consistently off in one direction.</li>
    <li>The <b>variance</b> is the average of the squared distances from the mean. It measures how spread out the readings are, in units of C<sup>2</sup>.</li>
    <li>The <b>standard deviation</b> is the square root of the variance, back in C. It is the typical size of the noise.</li>
</ul>

    $ make noise_stats
    Samples:             200
    Mean:                20.025 C  (bias 0.025 C)
    Standard deviation:  0.727 C
    Variance (R):        0.528 C^2

    Within 1 std dev:     71.0%
    Within 2 std dev:     96.0%
    Within 3 std dev:     99.5%

The bias is tiny, so this sensor is <b>accurate</b> on average, but each reading wanders by about 0.73 C, so any single reading is not very <b>precise</b>. This is the difference between precision and accuracy from the start of the semester, measured on a real data set.

The last three lines are the important ones. Noise that follows the familiar bell curve, a <b>Gaussian</b> distribution, puts about 68% of readings within one standard deviation, 95% within two, and 99.7% within three. Our sensor is close to that. So a reading more than 3 standard deviations from where we expect it should happen only about 3 times in 1000. If it happens, it is worth a second look. That idea comes back in Reading 38-2.

The variance gets a name, <b>R</b>, because the Kalman filter in Reading 38-2 uses it directly.

### Two Simple Filters - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec38/smoothing.c">smoothing.c</a>

<code>mission.csv</code> records a temperature that slowly rises, then jumps by about 1.6 C at sample 36, perhaps because a heater turned on. Each line has the sample number, the true temperature, and what the sensor measured. Here are two common ways to smooth the measurements:
<ul>
    <li>A <b>moving average</b> reports the average of the last 8 readings, kept in a ring buffer like the event log from Lecture 37.</li>
    <li><b>Exponential smoothing</b> keeps one number and moves it a fraction <code>ALPHA</code> of the way toward each new reading:</li>
</ul>

    ema = ema + ALPHA * (meas - ema);

    $ make smoothing
    sample     true measured moving avg exponential
        35    22.80    22.56      22.90      22.74
        36    24.46    25.13      23.07      23.22
        37    24.52    24.36      23.30      23.45
        38    24.58    24.38      23.57      23.63
        39    24.64    24.62      23.74      23.83
        40    24.70    25.23      24.02      24.11
          ...
        43    24.88    26.21      24.74      24.52

    RMS error over 60 samples:
      measured:     1.078 C
      moving avg:   0.499 C
      exponential:  0.458 C

Both filters cut the error roughly in half. <b>RMS</b> (root mean square) error is the square root of the average squared error, the same kind of measure as the standard deviation.

But look at sample 36. The true temperature jumped to 24.46 C, and both filters were still reporting about 23 C. The moving average needs about 8 samples to catch up completely, because it takes 8 new readings to flush out the old ones.

<b>This is the trade-off every filter makes.</b> Averaging more readings reduces noise, but makes the filter slower to notice a real change. A smaller <code>ALPHA</code> or a longer window gives a smoother output and more lag. On a spacecraft, lag can be dangerous: a filtered temperature that is 1.5 C behind reality can delay a fault response by many seconds.

How do you choose <code>ALPHA</code>? So far, by guessing. The Kalman filter replaces the guess with a calculation.

### <font color = "red">Class Introduction Question #1 - Using the output of <code>noise_stats</code>, is this sensor accurate, precise, both, or neither? Explain using the bias and the standard deviation.</font>

### <font color = "red">Class Introduction Question #2 - If the noise is Gaussian, about how often should a reading land more than 3 standard deviations from the expected value? Why does that make a 3-standard-deviation reading worth a second look?</font>

### <font color = "red">Class Introduction Question #3 - Both filters cut the RMS error in half. What did they cost? Using the table around sample 36, explain the trade-off.</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec38/Reading%2038-2.ipynb">Reading 38-2 - The Kalman Filter</a>